In [7]:
import torch

from torch import nn
from torch.utils.data import DataLoader, IterableDataset

import spacy

import pandas as pd

In [8]:
# Previously downloaded using terminal
# python -m spacy download es_core_web_sm
# python -m spacy download en_core_web_sm

nlp_es = spacy.load("es_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

doc = nlp_es("Hola mundo.")
tokens = [token.text for token in doc]

In [9]:
class LoadLanguageData(IterableDataset):
    
    def __init__(self, path:str, chunk_size:int = 1000) -> None:
    
        super().__init__()
        self.path = path
        self.chunk_size = chunk_size

    def __iter__(self):

        chunk_reader = pd.read_csv(
            self.path,
            sep="\t",
            chunksize=self.chunk_size ,
            header=None,
            usecols=[0, 1]
        )

        for current_chunk in chunk_reader:
            for row in current_chunk.itertuples(index=False, name=None):
                yield row


class TokenizerMachineTranslation:

    def __init__(self, source_dataset:LoadLanguageData, model_names:tuple = ("es_core_news_sm", "en_core_web_sm")):

        self.source = source_dataset

        self.nlp_language1 = spacy.load(model_names[0], disable=["parser", "ner", "lemmatizer"])
        self.nlp_language2 = spacy.load(model_names[1], disable=["parser", "ner", "lemmatizer"])

        self.word2index_language1 = {"[PAD]":0, "[START]":1, "[END]":2, "[UNK]":3}
        self.word2index_language2 = {"[PAD]":0, "[START]":1, "[END]":2, "[UNK]":3}

        self.state_build_vocab = False
        self.max_length = 0


    def __process_batch(self, texts_l1, texts_l2):
        # nlp.pipe es un generador, procesa de forma eficiente en paralelo
        # n_process=-1 usa todos los núcleos disponibles (solo en Linux/macOS)
        for doc in self.nlp_language1.pipe(texts_l1, batch_size=len(texts_l1)):
            for token in doc:
                if token.text not in self.word2index_language1:
                    idx = len(self.word2index_language1)
                    self.word2index_language1[token.text] = idx

        for doc in self.nlp_language2.pipe(texts_l2, batch_size=len(texts_l2)):
            for token in doc:
                if token.text not in self.word2index_language2:
                    idx = len(self.word2index_language2)
                    self.word2index_language2[token.text] = idx


    def build_vocabulary(self):
        """
        Consume el generador por bloques para aprovechar nlp.pipe
        """
        # Recolectamos textos en buffers para nlp.pipe
        buffer_l1 = []
        buffer_l2 = []
        
        for text_l1, text_l2 in self.source:
            buffer_l1.append(text_l1.lower())
            buffer_l2.append(text_l2.lower())

            # Cuando el buffer alcanza el tamaño del chunk, procesamos en masa
            if len(buffer_l1) >= self.source.chunk_size:
                self.__process_batch(buffer_l1, buffer_l2)
                buffer_l1, buffer_l2 = [], [] # Limpiamos buffers

        # Procesar remanentes si quedan
        if buffer_l1:
            self.__process_batch(buffer_l1, buffer_l2)
        
        self.state_build_vocab = True

    def __len__(self):
        return 1000

    def __iter__(self):
        if not self.state_build_vocab:
            raise AttributeError("Vocabulary has not been constructed. Run build_vocabulary()")

        for row in self.source:
            yield row

In [10]:
translation_iter = LoadLanguageData(path="data/spa-eng/spa.txt")
tokenizer = TokenizerMachineTranslation(translation_iter)

tokenizer.build_vocabulary()

In [11]:
translation_loader = DataLoader(tokenizer, batch_size=64, num_workers=1, shuffle=False)

In [12]:
for i in translation_loader:
    print(i)
    break

Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/multiprocessing/queues.py", line 239, in _feed
    reader_close()
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/multiprocessing/queues.py", line 271, in 

TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/_utils/worker.py", line 349, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lromero/mambaforge/envs/pytorch_env/lib/python3.11/site-packages/torch/utils/data/_utils/fetch.py", line 52, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
TypeError: 'TokenizerMachineTranslation' object is not subscriptable
